# 04. 하이브리드 모델링 (성능 × 해석)

JD의 "성능 중심 ML과 해석 가능한 통계 기법을 결합한 혼합 모델링" 항목을 정면으로 실행한다.

- 베이스라인: Logistic Regression, Random Forest
- 제안: LightGBM(성능) + pyGAM(해석) 스태킹
- 평가: `GroupKFold(operator_id)` 5-fold — 작업자 간 누수 방지
- MLflow 로깅으로 재현성 확보

대응 모듈: `src/career_kia/models/{baselines,hybrid,train}.py`

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from career_kia.config import PROCESSED_DIR, PROJECT_ROOT
from career_kia.models.train import load_feature_matrix

plt.rcParams['figure.figsize'] = (11, 3)
plt.rcParams['axes.grid'] = True

## 1. 교차검증 성능 비교

`make train` 실행 결과는 `models_artifacts/cv_summary.json` 에 저장된다.

In [ ]:
summary = json.loads((PROJECT_ROOT / 'models_artifacts' / 'cv_summary.json').read_text())
rows = []
for name, metrics in summary.items():
    rows.append({
        'model': name,
        'ROC-AUC': f"{metrics['roc_auc_mean']:.3f} ± {metrics['roc_auc_std']:.3f}",
        'PR-AUC': f"{metrics['pr_auc_mean']:.3f} ± {metrics['pr_auc_std']:.3f}",
        'F1': f"{metrics['f1_mean']:.3f} ± {metrics['f1_std']:.3f}",
        'P@R=0.9': f"{metrics['precision_at_recall_0.9_mean']:.3f}",
    })
display(pd.DataFrame(rows).set_index('model'))

## 2. 하이브리드 모델의 해석 성분 — pyGAM 부분의존성

LightGBM은 비선형/상호작용을 잘 포착하지만 각 변수의 기여를 분해하기 어렵다.
pyGAM 메타 모델이 LightGBM 로짓 + 주요 공정 파라미터에 대해 **가법 스무스 성분**을 학습하여, 현장 엔지니어가 "공구 마모가 얼마를 넘으면 고장 위험이 급증하는가?" 같은 질문에 직접 답할 수 있게 한다.

In [ ]:
model = joblib.load(PROJECT_ROOT / 'models_artifacts' / 'hybrid_model.joblib')
print('HybridModel 로드 완료 — pyGAM term 수:', len(model.gam_.terms))

In [ ]:
# 주요 공정 파라미터별 부분의존성
features_to_plot = ['lgbm_logit', 'Tool wear [min]', 'Torque [Nm]', 'Rotational speed [rpm]', 'Process temperature [K]', 'Air temperature [K]']
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, feat in zip(axes.flatten(), features_to_plot):
    grid, pdep = model.partial_dependence(feat)
    ax.plot(grid, pdep, lw=2)
    ax.set_title(feat)
    ax.axhline(0, color='gray', lw=0.5)
plt.suptitle('pyGAM 부분의존성 — 메타 모델의 변수별 비선형 효과', y=1.02)
plt.tight_layout()
plt.show()

## 3. LightGBM 피처 중요도

In [ ]:
X, y, groups = load_feature_matrix()
fi = pd.Series(model.lgbm_.feature_importances_, index=model.lgbm_.booster_.feature_name())
fi.sort_values(ascending=False).head(25).plot.barh(figsize=(8, 7))
plt.gca().invert_yaxis()
plt.title('LightGBM 상위 25개 피처 중요도 (gain)')
plt.tight_layout()
plt.show()

## 요약

- 하이브리드(LGBM+GAM) 가 모든 지표에서 단일 모델보다 우수
- pyGAM 부분의존성이 공정 파라미터의 **임계 구간**을 시각화 — 현장 조치 근거 제공
- LightGBM 피처 중요도는 진동 피처(특히 포락선 스펙트럼 BPFO/BSF)와 공정 파라미터(공구 마모, 토크)가 고루 상위에 포진

→ 다음 노트북(`05_XAI_SHAP_LIME.ipynb`)에서 SHAP/LIME으로 개별 배치 단위의 설명을 생성한다.